# 02 Generování oken z genomu (úkol 1.3)

Rozřeže genom na okna o délce `WINDOW`, vyřadí okna s více než 1 % N a rozdělí je
podle chromozomů: **trénink** chr1–22 a X kromě chr20 a chr21, **validace** chr20,
**test** chr21. Ukládají se jen souřadnice oken a jejich GC obsah, podíl N a repetic,
sekvence se vyřezávají až při tokenizaci. Stačí CPU runtime, trvá pár minut.

Předpoklad: hotový notebook 01.

## 1. Nastavení

In [ ]:
WINDOW = 1024          # délka okna v bp
MAX_N_FRAC = 0.01      # max. podíl N v okně

## Prostředí

Funguje v Colabu (prohlížeč i VS Code) i lokálně. V Colabu připojí Google Drive,
stáhne kód z GitHubu a zkopíruje genom na rychlý lokální disk. Jinde se cesty
řídí proměnnou `DNALM_ROOT`. Všechny cesty jsou v objektu `P` (`P.fasta`,
`P.windows("train")`, `P.tokenizer("bpe_600")`, `P.figures`, ...).

Ve VS Code s Colab rozšířením: pokud se Drive nepřipojí, použij příkaz
`Colab: Mount Google Drive to Server...` a buňku spusť znovu.

In [ ]:
# Úvodní buňka (stejná ve všech notebookách, neměnit).
# Colab: stáhne aktuální kód z GitHubu. Lokálně (Mac, školní GPU): použije nainstalované repo.
import importlib.util, sys
if importlib.util.find_spec("google.colab"):
    !git clone -q https://github.com/Valasik0/dna-lm.git /content/dna-lm 2>/dev/null || git -C /content/dna-lm pull -q
    !git -C /content/dna-lm log --oneline -1
    if "/content/dna-lm/src" not in sys.path:
        sys.path.insert(0, "/content/dna-lm/src")

from dnalm.utils.colab import setup
P = setup(window=WINDOW, genome=True)

## 2. Generování oken

In [ ]:
import time
from dnalm.data.genome import WindowConfig, make_windows

cfg = WindowConfig(window=WINDOW, max_n_frac=MAX_N_FRAC)
t0 = time.time()
summary = make_windows(str(P.fasta), str(P.windows_dir), cfg)
print(f"Hotovo za {time.time() - t0:.0f} s")

## 3. Přehled

In [ ]:
import pandas as pd
from dnalm.data.genome import load_windows

rows = []
for split, info in summary["splits"].items():
    df = load_windows(P.windows(split))
    chroms = info["chromosomes"]
    rows.append({
        "split": split,
        "chromozomy": ", ".join(chroms) if len(chroms) < 3 else f"{len(chroms)} chromozomů",
        "posun_bp": info["stride"],
        "oken": len(df),
        "vyřazeno_N": info["windows_total"] - info["windows_kept"],
        "GC_průměr": round(df.gc_frac.mean(), 3),
        "repetice_průměr": round(df.repeat_frac.mean(), 3),
    })
overview = pd.DataFrame(rows).set_index("split")
overview.to_csv(P.windows_dir / "overview.csv")
overview

## 4. Kontrola: sekvence několika oken

In [ ]:
from dnalm.data.genome import GenomeWindows

val = GenomeWindows(str(P.fasta), str(P.windows("val")))
print(len(val), "validačních oken")
for i in [0, len(val) // 2, len(val) - 1]:
    s = val[i]
    print(val.chrom[i], val.start[i], val.end[i], len(s), s[:60] + "...")
print("reverzní komplement prvního okna:", val.get(0, reverse=True)[:60] + "...")